# MethyAgent — 示例运行演示

**双智能体甲基化数据采集系统 (MethyAgent) 端到端演示**

本 notebook 演示以下完整流程：
1. 查询解析（中英文语义 + 精确 accession 识别）
2. Agent 1 (DatabaseAgent)：GEO + TCGA 数据库搜索与下载
3. Agent 2 (LiteratureAgent)：PubMed/PMC/bioRxiv 文献挖掘
4. 双 Agent 去重协调（共享 Registry）
5. **三层 LLM 辅助 PDF 提取流水线**（正则 → LLM → GEO API 验证）
6. 最终报告生成

> **查询示例**：`"EPIC平台在2024年的乳腺癌相关甲基化数据"`


## 0. 环境初始化

In [ ]:
import sys, os, json, re, time
from unittest.mock import MagicMock, patch

# 将 methyagent 包路径加入 sys.path
sys.path.insert(0, "/mnt/results/methyagent")

print("Python:", sys.version.split()[0])
print("MethyAgent 路径:", "/mnt/results/methyagent")
print("\n已加载模块:")
modules = [
    "registry.registry",
    "tools.parser_tools",
    "tools.pdf_section_extractor",
    "tools.llm_accession_extractor",
    "tools.geo_tools",
]
for m in modules:
    print(f"  ✓ {m}")


## 1. 初始化共享 Registry

两个 Agent 共用同一个 SQLite 数据库，通过 Registry 实现去重协调。

In [ ]:
from registry.registry import Registry

# 使用 /workspace 本地路径（避免 S3 随机写入限制）
registry = Registry("/workspace/demo_methyagent.db")

print("✓ Registry 初始化完成")
print(f"  数据库路径: /workspace/demo_methyagent.db")
print(f"  表结构:")
print(f"    - datasets              (主表：accession、状态、元数据)")
print(f"    - download_log          (下载事件日志)")
print(f"    - llm_extraction_cache  (LLM 提取结果缓存，按 DOI 索引)")
print(f"\n  新增字段 (LLM 扩展):")
print(f"    - needs_review    INTEGER  (1=待人工审核)")
print(f"    - llm_evidence    TEXT     (LLM 提取的原文证据)")


## 2. 查询解析 (parse_query node)

支持中英文语义查询和精确 accession 号输入。

In [ ]:
from tools.parser_tools import extract_accessions

print("=" * 60)
print("查询解析示例")
print("=" * 60)

queries = [
    "EPIC平台在2024年的乳腺癌相关甲基化数据",
    "下载GEO编号GSE124600的所有数据",
    "breast cancer WGBS methylation 2023",
]

for q in queries:
    accs = extract_accessions(q)
    geo_hits = accs.get("geo", [])
    print(f"\n  输入: {q}")
    if geo_hits:
        print(f"  → 直接识别 accession: {geo_hits}")
        print(f"  → 模式: exact_accession")
    else:
        print(f"  → 模式: semantic_search")
        cancer = "Breast Neoplasms" if ("乳腺癌" in q or "breast cancer" in q) else "Unknown"
        platform = "EPIC" if "EPIC" in q else ("WGBS" if "WGBS" in q else None)
        m = re.search(r'(?<!\d)(20\d{2})(?!\d)', q)
        year = int(m.group(1)) if m else None
        print(f"  → 解析结果: cancer={cancer}, platform={platform}, year={year}")


## 3. Agent 1 — DatabaseAgent

搜索 GEO 和 TCGA 数据库，注册发现的数据集，执行下载。

In [ ]:
print("=" * 60)
print("Agent 1 — DatabaseAgent (GEO + TCGA 搜索)")
print("=" * 60)

# 模拟 GEO 搜索结果（真实运行时由 GEOClient.search_gse() 返回）
mock_geo_results = [
    {
        "accession": "GSE225845",
        "title": "DNA methylation profiling of breast cancer using EPIC array (2024)",
        "platforms": ["GPL21145"],
        "platform_canonical": "EPIC",
        "sample_count": 120,
        "data_type": "array",
        "year": 2024,
        "source": "GEO",
        "supplementary_files": [
            "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE225nnn/GSE225845/suppl/GSE225845_beta_values.txt.gz"
        ]
    },
    {
        "accession": "GSE198978",
        "title": "EPIC methylation array in triple-negative breast cancer",
        "platforms": ["GPL21145"],
        "platform_canonical": "EPIC",
        "sample_count": 85,
        "data_type": "array",
        "year": 2024,
        "source": "GEO",
        "supplementary_files": [
            "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE198nnn/GSE198978/suppl/GSE198978_methylation_matrix.csv.gz"
        ]
    },
    {
        "accession": "GSE201543",
        "title": "Breast cancer methylation landscape: EPIC 850K profiling",
        "platforms": ["GPL21145"],
        "platform_canonical": "EPIC",
        "sample_count": 210,
        "data_type": "array",
        "year": 2023,
        "source": "GEO",
        "supplementary_files": []
    },
]

mock_tcga_results = [
    {
        "accession": "TCGA-BRCA",
        "title": "TCGA Breast Invasive Carcinoma — Methylation 450K",
        "platform_canonical": "450K",
        "sample_count": 894,
        "data_type": "array",
        "year": 2012,
        "source": "TCGA",
    }
]

print(f"\n  [GEO 搜索] 查询: 'EPIC breast cancer methylation 2024'")
print(f"  → 找到 {len(mock_geo_results)} 个数据集\n")

for ds in mock_geo_results:
    print(f"  ┌─ {ds['accession']}")
    print(f"  │  标题: {ds['title'][:60]}...")
    print(f"  │  平台: {ds['platform_canonical']} | 样本数: {ds['sample_count']} | 年份: {ds['year']}")
    print(f"  │  补充文件: {len(ds['supplementary_files'])} 个")
    registry.upsert_dataset(
        accession=ds["accession"], source=ds["source"], discovered_by="agent1",
        data_type=ds["data_type"], platform=ds["platform_canonical"],
        sample_count=ds["sample_count"], year=ds["year"], title=ds["title"],
        download_status="pending",
    )
    registry.log_event(ds["accession"], "start", "Registered by DatabaseAgent")
    print(f"  └─ ✓ 已注册到 Registry (status=pending)\n")

print(f"  [TCGA 搜索] 查询: 'BRCA methylation'")
print(f"  → 找到 {len(mock_tcga_results)} 个项目\n")
for ds in mock_tcga_results:
    print(f"  ┌─ {ds['accession']}")
    print(f"  │  标题: {ds['title']}")
    print(f"  │  平台: {ds['platform_canonical']} | 样本数: {ds['sample_count']}")
    registry.upsert_dataset(
        accession=ds["accession"], source=ds["source"], discovered_by="agent1",
        data_type=ds["data_type"], platform=ds["platform_canonical"],
        sample_count=ds["sample_count"], year=ds["year"], title=ds["title"],
        download_status="pending",
    )
    registry.log_event(ds["accession"], "start", "Registered by DatabaseAgent")
    print(f"  └─ ✓ 已注册到 Registry (status=pending)\n")

# 模拟下载完成
registry.update_status("GSE225845", "done", local_path="./data/methylation/GSE225845/", file_size_bytes=120*850000)
registry.log_event("GSE225845", "done", "Downloaded by DatabaseAgent")
registry.update_status("GSE198978", "done", local_path="./data/methylation/GSE198978/", file_size_bytes=85*850000)
registry.log_event("GSE198978", "done", "Downloaded by DatabaseAgent")
registry.update_status("GSE201543", "failed")
registry.log_event("GSE201543", "error", "Connection timeout after 3 retries")
registry.update_status("TCGA-BRCA", "done", local_path="./data/methylation/TCGA-BRCA/", file_size_bytes=894*2_000_000)
registry.log_event("TCGA-BRCA", "done", "Downloaded by DatabaseAgent")

print("  Agent 1 下载结果:")
print(f"  ✓ GSE225845  — 下载完成 ({120*850000/1e6:.1f} MB)")
print(f"  ✓ GSE198978  — 下载完成 ({85*850000/1e6:.1f} MB)")
print(f"  ✗ GSE201543  — 下载失败 (Connection timeout after 3 retries)")
print(f"  ✓ TCGA-BRCA  — 下载完成 ({894*2/1e3:.1f} GB)")


## 4. Agent 2 — LiteratureAgent：文献搜索与摘要提取

搜索 PubMed/PMC/bioRxiv，从摘要中提取 accession 号。

In [ ]:
print("=" * 60)
print("Agent 2 — LiteratureAgent (文献挖掘)")
print("=" * 60)

mock_papers = [
    {
        "pmid": "38234567", "doi": "10.1038/s41586-024-07234-1", "pmc_id": "PMC10987654",
        "title": "Epigenome-wide methylation profiling reveals breast cancer subtypes",
        "abstract": "We performed EPIC array methylation profiling on 156 breast cancer samples. "
                    "Raw data are deposited in GEO under accession GSE234891. "
                    "Additional TCGA-BRCA data were used for validation.",
        "year": 2024, "journal": "Nature", "source": "pubmed",
    },
    {
        "pmid": "38198765", "doi": "10.1016/j.cell.2024.01.045", "pmc_id": "PMC10876543",
        "title": "Single-cell methylome analysis of breast cancer progression",
        "abstract": "RRBS sequencing was performed on 2,400 single cells. "
                    "Data available at GEO (GSE225845) and ArrayExpress (E-MTAB-12456).",
        "year": 2024, "journal": "Cell", "source": "pubmed",
    },
    {
        "pmid": "38156789", "doi": "10.1093/nar/gkad1234", "pmc_id": None,
        "title": "Comprehensive methylation atlas of breast cancer using 850K EPIC",
        "abstract": "We generated a comprehensive methylation atlas. "
                    "All data are available upon reasonable request.",
        "year": 2024, "journal": "Nucleic Acids Research", "source": "pubmed",
    },
    {
        "pmid": "38112345", "doi": "10.1101/2024.01.15.576012", "pmc_id": None,
        "title": "WGBS methylation landscape in HER2+ breast cancer",
        "abstract": "Whole genome bisulfite sequencing data deposited at GEO: GSE201543.",
        "year": 2024, "journal": "bioRxiv", "source": "biorxiv",
    },
]

print(f"\n  [PubMed 搜索] 查询: '"Breast Neoplasms"[MeSH] AND (EPIC OR HumanMethylationEPIC) AND DNA methylation[MeSH]'")
print(f"  → 找到 {len(mock_papers)} 篇论文\n")

for paper in mock_papers:
    accs = extract_accessions(paper["abstract"])
    paper["accessions"] = accs
    paper["has_accessions"] = any(accs[k] for k in ("geo", "tcga"))
    geo_found = accs.get("geo", [])
    tcga_found = accs.get("tcga", [])
    print(f"  ┌─ PMID {paper['pmid']} | {paper['journal']} {paper['year']}")
    print(f"  │  {paper['title'][:65]}...")
    if geo_found or tcga_found:
        print(f"  │  摘要中发现 accession: GEO={geo_found} TCGA={tcga_found}")
    else:
        print(f"  │  摘要中未发现 accession → 将尝试全文/PDF提取")
    print(f"  └─ DOI: {paper['doi']}\n")


## 5. 去重协调 — Agent 2 检查 Agent 1 的 Registry

 Agent 2 在下载前先查询 Registry，跳过已被 Agent 1 覆盖的数据集。

In [ ]:
print("=" * 60)
print("去重检查 — Agent 2 与 Agent 1 结果对比")
print("=" * 60)

existing = registry.get_accession_set()
print(f"\n  Registry 中已有 {len(existing)} 个 accession (来自 Agent 1):")
for acc in sorted(existing):
    rec = registry.get(acc)
    print(f"    {acc:20s} status={rec['download_status']:10s} by={rec['discovered_by']}")

print()

# 收集文献中的所有 accession
lit_accessions = []
for paper in mock_papers:
    accs = paper.get("accessions", {})
    pmid = paper.get("pmid")
    doi = paper.get("doi", "")
    for acc in accs.get("geo", []):
        if acc.startswith("GSE"):
            lit_accessions.append({"accession": acc, "pmid": pmid, "doi": doi})
    for acc in accs.get("tcga", []):
        lit_accessions.append({"accession": acc, "pmid": pmid, "doi": doi})

print(f"  文献中发现的 accession ({len(lit_accessions)} 个):")
new_accessions = []
skipped = []
for item in lit_accessions:
    acc = item["accession"]
    if acc in existing:
        print(f"    {acc:20s} → ⚠ 跳过 (已被 Agent 1 下载)")
        skipped.append(acc)
    else:
        print(f"    {acc:20s} → ✓ 新发现，加入下载队列")
        new_accessions.append(item)

print(f"\n  去重结果: {len(new_accessions)} 个新 accession, {len(skipped)} 个跳过")


## 6. 三层 LLM 流水线 — Layer 1+前置：PDFSectionExtractor

从 PDF 全文中精准定位目标章节，减少送给 LLM 的 token 数量。

In [ ]:
from tools.pdf_section_extractor import PDFSectionExtractor

print("=" * 60)
print("PDFSectionExtractor — 章节定位演示")
print("=" * 60)

# 模拟 NAR 论文的 PDF 文本（摘要中无 accession，需要从 Data Availability 节提取）
simulated_pdf_text = """
Abstract
We generated a comprehensive methylation atlas of breast cancer using the 850K EPIC array.
All data are available upon reasonable request.

Introduction
Breast cancer is the most common malignancy in women worldwide...

Materials and Methods
Sample collection: 312 primary breast tumor samples were collected.
DNA methylation profiling was performed using the Illumina EPIC array (850K).
Raw IDAT files were processed using minfi R package.

Data Availability
The methylation data generated in this study have been deposited in the Gene Expression
Omnibus (GEO) database under accession number GSE267834. Additional processed data
including beta value matrices are available at Figshare (doi: 10.6084/m9.figshare.24567890).
Accession codes: GSE267834 (GEO), E-MTAB-13456 (ArrayExpress).

Results
We identified 45,678 differentially methylated positions (DMPs) between tumor and normal tissue...

Discussion
Our findings suggest that EPIC array methylation profiling can distinguish breast cancer subtypes...
"""

extractor = PDFSectionExtractor()
result = extractor.extract(simulated_pdf_text)

print(f"\n  PDF 全文: {len(simulated_pdf_text)} 字符")
print(f"  找到章节: {len(result.sections)} 个")
print(f"  含触发词章节: {len(result.trigger_sections)} 个")
print()

for sec in result.sections:
    trigger_mark = "🎯" if sec.has_trigger else "  "
    print(f"  {trigger_mark} [{sec.section_name}] ({sec.char_count} chars)")
    if sec.has_trigger:
        preview = sec.text[:200].replace('\n', ' ')
        print(f"      预览: {preview}...")

print(f"\n  送给 LLM 的文本: {len(result.best_text)} 字符")
print(f"  节省 token: {100*(1-len(result.best_text)/len(simulated_pdf_text)):.0f}%")


## 7. 三层流水线 — Layer 1：正则提取

零成本，覆盖约 80% 的标准格式 accession。

In [ ]:
print("=" * 60)
print("Layer 1 — 正则提取")
print("=" * 60)

section_text = result.best_text  # 来自上一步的 Data Availability 节
regex_result = extract_accessions(section_text)
geo_found = regex_result.get("geo", [])
tcga_found = regex_result.get("tcga", [])
all_regex = list(set(geo_found + tcga_found))

print(f"\n  输入文本: {len(section_text)} 字符")
print(f"  GEO accessions: {geo_found}")
print(f"  TCGA accessions: {tcga_found}")

if all_regex:
    print(f"\n  → ✓ 正则成功！找到 {len(all_regex)} 个 accession，跳过 LLM")
    print(f"  → 结果: {all_regex}")
    print(f"  → 动作: 直接写入 Registry，触发下载")
else:
    print(f"\n  → ✗ 正则未找到 accession，触发 Layer 2 (LLM)")


## 8. 三层流水线 — Layer 2：LLM 提取

仅在正则失败时触发。支持中英文双语，输出三级置信度。

In [ ]:
from tools.llm_accession_extractor import LLMAccessionExtractor

print("=" * 60)
print("Layer 2 — LLM 提取（非标准中文描述场景）")
print("=" * 60)

# 极端案例：中文非标准描述，无法用正则提取
hard_case_text = """
数据可用性

本研究的甲基化测序数据已上传至公共数据库，
读者可通过联系通讯作者获取数据集访问权限，
相关数据存储于GEO平台，序列号为二零二三年提交的第二百八十九号数据集。
"""

print(f"  输入文本:")
print(f"  {hard_case_text.strip()}")

# Layer 1 先尝试
regex_hard = extract_accessions(hard_case_text)
print(f"\n  Layer 1 正则结果: GEO={regex_hard.get('geo', [])} → 空，触发 LLM")

# 模拟 LLM 响应（真实运行时调用 GPT-4o）
mock_llm = MagicMock()
mock_llm.invoke.return_value = MagicMock(content=json.dumps({
    "extractions": [
        {
            "accession": "GSE289000",
            "database": "GEO",
            "confidence": "medium",
            "evidence": "GEO平台，序列号为二零二三年提交的第二百八十九号数据集",
            "context": "甲基化测序数据，2023年提交，编号约为GSE289xxx"
        }
    ],
    "summary": "发现1个GEO数据集，置信度中等（accession号从中文描述推断）"
}))

extractor_llm = LLMAccessionExtractor(
    llm=mock_llm,
    cache_db_path="/workspace/demo_llm_cache.db",
    model_name="gpt-4o",
)

llm_result = extractor_llm.extract(hard_case_text, doi="10.1234/hard-case-demo")

print(f"\n  LLM 提取结果:")
print(f"  ├─ 模型: {llm_result.model_used}")
print(f"  ├─ 缓存命中: {llm_result.cache_hit}")
print(f"  ├─ high confidence (自动下载): {llm_result.high_confidence}")
print(f"  ├─ medium confidence (待审核): {llm_result.pending_review}")
print(f"  └─ 摘要: {llm_result.summary}")

if llm_result.extractions:
    e = llm_result.extractions[0]
    print(f"\n  提取详情:")
    print(f"  ├─ accession: {e.accession}")
    print(f"  ├─ 置信度: {e.confidence} → 动作: {e.action}")
    print(f"  └─ 证据: \"{e.evidence}\"")

# 演示缓存命中
print(f"\n  [缓存演示] 第二次调用相同 DOI:")
llm_result2 = extractor_llm.extract("completely different text", doi="10.1234/hard-case-demo")
print(f"  LLM 调用次数: {mock_llm.invoke.call_count} (未增加)")
print(f"  cache_hit: {llm_result2.cache_hit} ✓ 节省了 1 次 API 调用")


## 9. 三层流水线 — Layer 3：GEO API 验证

调用 NCBI E-utilities 验证 LLM 提取的 accession 是否真实存在，过滤幻觉。

In [ ]:
from tools.geo_tools import GEOClient

print("=" * 60)
print("Layer 3 — GEO API 验证（幻觉过滤）")
print("=" * 60)

client = GEOClient()

# 模拟 NCBI esearch 响应
def mock_get_verify(endpoint, params):
    term = params.get("term", "")
    mock_resp = MagicMock()
    mock_resp.raise_for_status = MagicMock()
    real_accessions = {"GSE267834", "GSE234891", "GSE289456", "GSE225845", "GSE198978"}
    found_real = any(acc in term for acc in real_accessions)
    mock_resp.json.return_value = {
        "esearchresult": {
            "count": "1" if found_real else "0",
            "idlist": ["12345"] if found_real else []
        }
    }
    return mock_resp

test_accessions = ["GSE267834", "GSE234891", "GSE999999", "GSE000001"]
print(f"\n  待验证 accession: {test_accessions}\n")

with patch.object(client, "_get", side_effect=mock_get_verify):
    for acc in test_accessions:
        exists = client.verify_accession(acc)
        status = "✓ 真实存在 → 写入 Registry" if exists else "✗ 不存在 (幻觉) → 丢弃"
        print(f"  NCBI esearch [{acc}]: {status}")

print(f"\n  批量验证（batch_verify_accessions）:")
print(f"  → 将多个 accession 合并为一次 OR 查询，减少 API 调用次数")
print(f"  → 示例: GSE267834[Accession] OR GSE234891[Accession] OR ...")


## 10. Agent 2 注册新发现数据集

高置信度自动下载，中置信度标记为 `pending_review` 等待人工确认。

In [ ]:
print("=" * 60)
print("Agent 2 — 注册新发现数据集")
print("=" * 60)

new_from_lit = [
    {
        "accession": "GSE234891", "source": "GEO", "pmid": "38234567",
        "doi": "10.1038/s41586-024-07234-1", "needs_review": False,
        "title": "Epigenome-wide methylation profiling reveals breast cancer subtypes",
        "sample_count": 156,
    },
    {
        "accession": "GSE267834", "source": "GEO", "pmid": "38156789",
        "doi": "10.1093/nar/gkad1234", "needs_review": False,
        "title": "Comprehensive methylation atlas of breast cancer using 850K EPIC",
        "sample_count": 312,
    },
    {
        "accession": "GSE289000", "source": "GEO", "pmid": None,
        "doi": "10.1234/hard-case-demo", "needs_review": True,
        "llm_evidence": "GEO平台，序列号为二零二三年提交的第二百八十九号数据集",
        "title": "LLM medium-confidence extraction",
        "sample_count": None,
    },
]

print()
for item in new_from_lit:
    acc = item["accession"]
    needs_review = item.get("needs_review", False)
    dl_status = Registry.STATUS_PENDING_REVIEW if needs_review else "pending"
    icon = "⏳" if needs_review else "✓"
    note = "(LLM medium confidence → 待人工审核)" if needs_review else "(LLM high confidence → 自动下载)"

    registry.upsert_dataset(
        accession=acc, source=item["source"], discovered_by="agent2",
        title=item["title"], paper_pmid=item.get("pmid"), paper_doi=item.get("doi"),
        sample_count=item.get("sample_count"), download_status=dl_status,
        needs_review=needs_review, llm_evidence=item.get("llm_evidence"),
    )
    registry.log_event(acc, "start", f"Registered by LiteratureAgent (needs_review={needs_review})")
    print(f"  {icon} {acc:20s} {note}")

# 模拟下载完成
registry.update_status("GSE234891", "done", local_path="./data/methylation/GSE234891/", file_size_bytes=156*850000)
registry.log_event("GSE234891", "done", "Downloaded by LiteratureAgent")
registry.update_status("GSE267834", "done", local_path="./data/methylation/GSE267834/", file_size_bytes=312*850000)
registry.log_event("GSE267834", "done", "Downloaded by LiteratureAgent")

print(f"\n  下载结果:")
print(f"  ✓ GSE234891 — 下载完成 ({156*850000/1e6:.1f} MB)")
print(f"  ✓ GSE267834 — 下载完成 ({312*850000/1e6:.1f} MB)")
print(f"  ⏳ GSE289000 — 等待人工审核 (不触发下载)")


## 11. 最终报告 (generate_report node)

汇总两个 Agent 的工作结果，展示完整的数据集清单和统计信息。

In [ ]:
print("=" * 60)
print("最终报告")
print("=" * 60)

summary = registry.get_summary()
all_datasets = registry.get_all()
pending_review = registry.get_pending_review()

print(f"""
  ╔══════════════════════════════════════════════════════╗
  ║           MethyAgent 运行报告                        ║
  ║  查询: EPIC平台2024年乳腺癌甲基化数据                ║
  ╚══════════════════════════════════════════════════════╝

  📊 数据集汇总
  ─────────────────────────────────────────────────────
  总计发现:          {summary['total']} 个数据集
  ├─ Agent 1 发现:   {summary['agent1_discovered']} 个 (GEO直接搜索 + TCGA)
  └─ Agent 2 发现:   {summary['agent2_discovered']} 个 (文献挖掘)""")

print("\n  📥 下载状态")
print("  ─────────────────────────────────────────────────────")
status_icons = {"done": "✓", "failed": "✗", "pending_review": "⏳", "pending": "○"}
for status, count in sorted(summary['by_status'].items()):
    icon = status_icons.get(status, "·")
    bar = "█" * count
    print(f"  {icon} {status:20s}: {count:3d}  {bar}")

print("\n  🗄️  数据来源")
print("  ─────────────────────────────────────────────────────")
for source, count in sorted(summary['by_source'].items()):
    print(f"      {source:20s}: {count:3d} 个")

print("\n  🤖 LLM 提取统计")
print("  ─────────────────────────────────────────────────────")
print(f"  LLM 缓存条目:      {summary['llm_cache_entries']} 个")
print(f"  待人工审核:        {summary['pending_review']} 个")

print("\n  📋 数据集详情")
print("  ─────────────────────────────────────────────────────")
for ds in all_datasets:
    icon = status_icons.get(ds['download_status'], '·')
    review_flag = " [需审核]" if ds.get('needs_review') else ""
    size_str = ""
    if ds.get('file_size_bytes'):
        mb = ds['file_size_bytes'] / 1e6
        size_str = f" ({mb:.0f} MB)" if mb < 1000 else f" ({mb/1000:.1f} GB)"
    print(f"  {icon} {ds['accession']:20s} by={ds['discovered_by']:10s} "
          f"status={ds['download_status']:15s}{size_str}{review_flag}")

if pending_review:
    print("\n  ⚠️  待审核数据集 (LLM medium confidence)")
    print("  ─────────────────────────────────────────────────────")
    for ds in pending_review:
        print(f"  ⏳ {ds['accession']}")
        print(f"     证据: {ds.get('llm_evidence', 'N/A')}")
        print(f"     批准: registry.approve_review('{ds['accession']}')")
        print(f"     拒绝: registry.reject_review('{ds['accession']}')")


## 12. 三层流水线效果对比

展示不同输入场景下各层的处理结果。

In [ ]:
print("=" * 60)
print("三层流水线效果对比")
print("=" * 60)

scenarios = [
    ("英文标准格式",   'Data deposited in GEO under accession GSE124600.'),
    ("中文标准格式",   '数据存储于GEO数据库，编号为GSE200234。'),
    ("英文隐式引用",   'methylation data are available at NCBI GEO (series GSE98765)'),
    ("中文非标准描述", '数据存储于GEO平台，序列号为二零二三年提交的第二百八十九号数据集'),
    ("无 accession",  'Data available upon reasonable request to the corresponding author.'),
]

print()
for label, text in scenarios:
    regex_result = extract_accessions(text)
    regex_found = regex_result.get("geo", []) + regex_result.get("tcga", [])
    print(f"  场景: {label}")
    print(f"  文本: \"{text[:70]}\"")
    if regex_found:
        print(f"  Layer 1 (正则): ✓ 找到 {regex_found} → 直接返回，跳过 LLM")
    else:
        print(f"  Layer 1 (正则): ✗ 未找到 → 触发 Layer 2 (LLM)")
        if "GEO" in text or "数据库" in text or "gse" in text.lower():
            print(f"  Layer 2 (LLM):  → 提取 medium confidence accession → pending_review")
            print(f"  Layer 3 (验证): → NCBI esearch 验证真实性")
        else:
            print(f"  Layer 2 (LLM):  → 返回空列表 (无数据库引用)")
    print()

print("  ─────────────────────────────────────────────────────")
print("  总结:")
print("  正则覆盖 ~80% 场景 (0 成本，毫秒级)")
print("  LLM 仅处理剩余 ~20% 非标准描述 (按需调用)")
print("  GEO API 验证过滤幻觉，确保数据质量")
print("  DOI 缓存避免重复 LLM 调用，跨会话持久化")


## 13. 人工审核操作

对 `pending_review` 数据集执行批准或拒绝操作。

In [ ]:
print("=" * 60)
print("人工审核操作演示")
print("=" * 60)

pending = registry.get_pending_review()
print(f"\n  当前待审核数据集: {len(pending)} 个")
for ds in pending:
    print(f"  ⏳ {ds['accession']}: {ds.get('llm_evidence', '')}")

# 批准 GSE289000
print(f"\n  执行: registry.approve_review('GSE289000')")
registry.approve_review("GSE289000")

rec = registry.get("GSE289000")
print(f"  审核后状态: needs_review={rec['needs_review']}, download_status={rec['download_status']}")
print(f"  → 状态变为 'pending'，可触发下载")

# 验证 pending_review 列表已清空
pending_after = registry.get_pending_review()
print(f"\n  审核后待审核数量: {len(pending_after)} 个")
print(f"  ✓ 审核流程完成")
